<a href="https://colab.research.google.com/github/NiharikaDas29/FinGuard-Q/blob/main/notebooks/finguard_data_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

import pandas as pd

filename = "PS_20174392719_1491204439457_log.csv"

# Load the CSV and inspect its structure
df = pd.read_csv(filename)

print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst five rows:")
display(df.head())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nFraud-label distribution:")
print(df["isFraud"].value_counts(dropna=False))

print("\nFraud percentages:")
print((df["isFraud"].value_counts(normalize=True) * 100).round(4))


Dataset shape: (28297, 11)

Column names:
['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']

First five rows:


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0.0,0.0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0.0,0.0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1.0,0.0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1.0,0.0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0.0,0.0



Data types:
step                int64
type               object
amount            float64
nameOrig           object
oldbalanceOrg     float64
newbalanceOrig    float64
nameDest           object
oldbalanceDest    float64
newbalanceDest    float64
isFraud           float64
isFlaggedFraud    float64
dtype: object

Missing values:
step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           1
isFlaggedFraud    1
dtype: int64

Fraud-label distribution:
isFraud
0.0    28212
1.0       84
NaN        1
Name: count, dtype: int64

Fraud percentages:
isFraud
0.0    99.7031
1.0     0.2969
Name: proportion, dtype: float64


In [3]:

import pandas as pd
import numpy as np

# Preserve the original dataframe and create a working copy
df_clean = df.copy()

# Remove rows without a known target label
missing_labels = df_clean["isFraud"].isna().sum()
df_clean = df_clean.dropna(subset=["isFraud"]).copy()

# Validate the target values
print("Missing target labels removed:", missing_labels)
print("Remaining rows:", len(df_clean))
print("Target values:", sorted(df_clean["isFraud"].unique()))

assert set(df_clean["isFraud"].unique()).issubset({0, 1})

df_clean["isFraud"] = df_clean["isFraud"].astype(int)

# Check duplicate rows and transaction identifiers
print("\nExact duplicate rows:", df_clean.duplicated().sum())
print("Duplicate transaction records are not removed automatically.")

print("\nFraud distribution:")
print(df_clean["isFraud"].value_counts())

print("\nFraud rate (%):",
      round(df_clean["isFraud"].mean() * 100, 4))

print("\nTransaction types:")
print(df_clean["type"].value_counts())

print("\nTime-step range:",
      df_clean["step"].min(), "to", df_clean["step"].max())


Missing target labels removed: 1
Remaining rows: 28296
Target values: [np.float64(0.0), np.float64(1.0)]

Exact duplicate rows: 0
Duplicate transaction records are not removed automatically.

Fraud distribution:
isFraud
0    28212
1       84
Name: count, dtype: int64

Fraud rate (%): 0.2969

Transaction types:
type
PAYMENT     13913
CASH_OUT     6108
CASH_IN      5027
TRANSFER     2735
DEBIT         513
Name: count, dtype: int64

Time-step range: 1 to 8


In [4]:

# Split chronologically using the 80th percentile of time steps
cutoff = df_clean["step"].quantile(0.80)

train_df = df_clean[df_clean["step"] < cutoff].copy()
test_df = df_clean[df_clean["step"] >= cutoff].copy()

print("Time cutoff:", cutoff)
print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))

print("\nTraining fraud counts:")
print(train_df["isFraud"].value_counts())

print("\nTesting fraud counts:")
print(test_df["isFraud"].value_counts())

print("\nTraining time range:",
      train_df["step"].min(), "to", train_df["step"].max())
print("Testing time range:",
      test_df["step"].min(), "to", test_df["step"].max())

assert train_df["step"].max() < test_df["step"].min()


Time cutoff: 8.0
Training rows: 14001
Testing rows: 14295

Training fraud counts:
isFraud
0    13923
1       78
Name: count, dtype: int64

Testing fraud counts:
isFraud
0    14289
1        6
Name: count, dtype: int64

Training time range: 1 to 7
Testing time range: 8 to 8


In [5]:

# Inspect fraud distribution by time step
step_summary = (
    df_clean.groupby("step")
    .agg(
        transactions=("isFraud", "size"),
        fraud_cases=("isFraud", "sum")
    )
    .reset_index()
)

step_summary["fraud_rate_pct"] = (
    100 * step_summary["fraud_cases"] / step_summary["transactions"]
).round(3)

display(step_summary)

print("\nFraud counts by step:")
print(df_clean.groupby("step")["isFraud"].sum())


,step,transactions,fraud_cases,fraud_rate_pct
0,1,2708,16,0.591
1,2,1014,8,0.789
2,3,552,4,0.725
3,4,565,10,1.770
4,5,665,6,0.902
5,6,1660,22,1.325
6,7,6837,12,0.176
7,8,14295,6,0.042



Fraud counts by step:
step
1    16
2     8
3     4
4    10
5     6
6    22
7    12
8     6
Name: isFraud, dtype: int64


In [6]:

from sklearn.model_selection import train_test_split

# Features available before the fraud label is known
FEATURES = ["step", "type", "amount"]

X = df_clean[FEATURES].copy()
y = df_clean["isFraud"].astype(int).copy()

# Preserve the original fraud proportion in both subsets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("STRATIFIED SPLIT SUMMARY")
print("-" * 40)
print("Total records:", len(X))
print("Training records:", len(X_train))
print("Testing records:", len(X_test))

print("\nTraining labels:")
print(y_train.value_counts())
print("Training fraud rate:", round(y_train.mean() * 100, 4), "%")

print("\nTesting labels:")
print(y_test.value_counts())
print("Testing fraud rate:", round(y_test.mean() * 100, 4), "%")

# Verify that both sets contain fraud examples
assert y_train.nunique() == 2
assert y_test.nunique() == 2

print("\nPASS: both subsets contain fraud and non-fraud records.")


STRATIFIED SPLIT SUMMARY
----------------------------------------
Total records: 28296
Training records: 21222
Testing records: 7074

Training labels:
isFraud
0    21159
1       63
Name: count, dtype: int64
Training fraud rate: 0.2969 %

Testing labels:
isFraud
0    7053
1      21
Name: count, dtype: int64
Testing fraud rate: 0.2969 %

PASS: both subsets contain fraud and non-fraud records.


In [7]:

import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

# Numerical and categorical features
NUMERIC_FEATURES = ["step", "amount"]
CATEGORICAL_FEATURES = ["type"]

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), NUMERIC_FEATURES),
        ("categorical", OneHotEncoder(handle_unknown="ignore"),
         CATEGORICAL_FEATURES)
    ]
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

# Train the model
model.fit(X_train, y_train)

# Predict fraud probabilities on held-out records
y_proba = model.predict_proba(X_test)[:, 1]

# Initial threshold: 0.5
y_pred = (y_proba >= 0.5).astype(int)

print("BASELINE MODEL RESULTS")
print("-" * 45)
print("PR-AUC (Average Precision):",
      round(average_precision_score(y_test, y_proba), 4))
print("Precision:", round(precision_score(y_test, y_pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, y_pred, zero_division=0), 4))
print("F1-score:", round(f1_score(y_test, y_pred, zero_division=0), 4))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))


BASELINE MODEL RESULTS
---------------------------------------------
PR-AUC (Average Precision): 0.0324
Precision: 0.0165
Recall: 0.9048
F1-score: 0.0324

Confusion matrix:
[[5921 1132]
 [   2   19]]

Classification report:
              precision    recall  f1-score   support

           0       1.00      0.84      0.91      7053
           1       0.02      0.90      0.03        21

    accuracy                           0.84      7074
   macro avg       0.51      0.87      0.47      7074
weighted avg       1.00      0.84      0.91      7074



In [8]:

# Build a ranked candidate queue using model scores only.
# Do NOT use y_test to select candidates.

import pandas as pd

# Risk scores correspond to the rows in X_test.
candidate_queue = X_test[["step", "type", "amount"]].copy()
candidate_queue["risk_score"] = y_proba

# Preserve a reference to the source row for traceability.
candidate_queue["transaction_id"] = candidate_queue.index

# Illustrative review-time assumptions; refine later if evidence is available.
review_time_map = {
    "TRANSFER": 3,
    "CASH_OUT": 3,
    "CASH_IN": 2,
    "PAYMENT": 1,
    "DEBIT": 1
}

candidate_queue["review_time"] = (
    candidate_queue["type"].map(review_time_map).fillna(2).astype(int)
)

# Convert risk scores to an integer priority from 1 to 100.
# The score is a ranking heuristic, not a calibrated fraud probability.
candidate_queue["priority"] = (
    1 + 99 * candidate_queue["risk_score"]
).round().astype(int)

candidate_queue = candidate_queue.sort_values(
    "risk_score", ascending=False
).reset_index(drop=True)

print("Top 20 candidate transactions:")
display(
    candidate_queue[
        [
            "transaction_id", "step", "type", "amount",
            "risk_score", "priority", "review_time"
        ]
    ].head(20)
)

print("\nTotal candidate records:", len(candidate_queue))
print("Fraud labels were not used to rank or select candidates.")


Top 20 candidate transactions:


,transaction_id,step,type,amount,risk_score,priority,review_time
0,375,1,TRANSFER,2545478.01,0.992211,99,3
1,1202,1,TRANSFER,2258388.15,0.992002,99,3
2,2591,1,TRANSFER,2032275.62,0.991833,99,3
3,1789,1,TRANSFER,1918461.70,0.991747,99,3
4,1785,1,TRANSFER,1636507.63,0.991529,99,3
5,362,1,TRANSFER,1457213.54,0.991388,99,3
6,800,1,TRANSFER,1440296.14,0.991374,99,3
7,1783,1,TRANSFER,1212719.61,0.991191,99,3
8,1229,1,TRANSFER,1201716.66,0.991182,99,3
9,2588,1,TRANSFER,1147953.82,0.991138,99,3



Total candidate records: 7074
Fraud labels were not used to rank or select candidates.


In [9]:

# Create a small, diverse shortlist for the first QAOA integration test.
# This does not use fraud labels.

shortlist = candidate_queue[
    [
        "transaction_id",
        "step",
        "type",
        "amount",
        "risk_score",
        "review_time"
    ]
].copy()

# Remove duplicate risk scores only for shortlist diversity.
shortlist = shortlist.drop_duplicates(
    subset=["risk_score", "review_time", "type"]
)

# Take a broader top-ranked shortlist first.
shortlist = shortlist.head(30).copy()

# Scale risk scores to a more useful range for optimization.
score_min = shortlist["risk_score"].min()
score_max = shortlist["risk_score"].max()

if score_max > score_min:
    shortlist["priority"] = (
        1 + 99 * (shortlist["risk_score"] - score_min)
        / (score_max - score_min)
    ).round().astype(int)
else:
    shortlist["priority"] = 50

print("Shortlist size:", len(shortlist))
display(shortlist.head(15))


Shortlist size: 30


,transaction_id,step,type,amount,risk_score,review_time,priority
0,375,1,TRANSFER,2545478.01,0.992211,3,100
1,1202,1,TRANSFER,2258388.15,0.992002,3,88
2,2591,1,TRANSFER,2032275.62,0.991833,3,78
3,1789,1,TRANSFER,1918461.70,0.991747,3,73
4,1785,1,TRANSFER,1636507.63,0.991529,3,61
5,362,1,TRANSFER,1457213.54,0.991388,3,53
6,800,1,TRANSFER,1440296.14,0.991374,3,52
7,1783,1,TRANSFER,1212719.61,0.991191,3,42
8,1229,1,TRANSFER,1201716.66,0.991182,3,41
9,2588,1,TRANSFER,1147953.82,0.991138,3,39


In [10]:

# Choose three real candidate transactions for the first integration test.
# We use model scores only, not the fraud labels.

qaoa_candidates = shortlist.iloc[[0, 5, 10]].copy()

qaoa_candidates = qaoa_candidates.reset_index(drop=True)
qaoa_candidates["case"] = ["A", "B", "C"]

# Keep only the fields needed by the optimizer.
qaoa_input = qaoa_candidates[
    [
        "case",
        "transaction_id",
        "risk_score",
        "priority",
        "review_time"
    ]
].copy()

print("FinGuard-Q optimization input:")
display(qaoa_input)

print("\nPriority values:", qaoa_input["priority"].tolist())
print("Review times:", qaoa_input["review_time"].tolist())

# Export for later use in Qollab or the final dashboard.
qaoa_input.to_csv("finguard_qaoa_candidates.csv", index=False)
print("\nSaved: finguard_qaoa_candidates.csv")


FinGuard-Q optimization input:


,case,transaction_id,risk_score,priority,review_time
0,A,375,0.992211,100,3
1,B,362,0.991388,53,3
2,C,379,0.990890,25,3



Priority values: [100, 53, 25]
Review times: [3, 3, 3]

Saved: finguard_qaoa_candidates.csv


In [11]:

from itertools import product

# Actual candidate data exported from our pipeline
cases = ["A", "B", "C"]
transaction_ids = [375, 362, 379]
priorities = [100, 53, 25]
review_times = [3, 3, 3]
budget = 6


def evaluate(bits):
    score = sum(p * x for p, x in zip(priorities, bits))
    time_used = sum(t * x for t, x in zip(review_times, bits))
    return score, time_used, time_used <= budget


feasible = []

for bits in product([0, 1], repeat=3):
    score, time_used, ok = evaluate(bits)

    if ok:
        selected = [
            cases[i] for i, x in enumerate(bits) if x
        ]
        feasible.append((score, time_used, selected, bits))

best = max(feasible, key=lambda row: row[0])

print("REAL-DATA OPTIMIZATION VALIDATION")
print("-" * 45)

for score, time_used, selected, bits in sorted(
    feasible, key=lambda row: row[0], reverse=True
):
    print(
        f"Selected: {selected or ['None']} | "
        f"Priority: {score} | Time: {time_used}"
    )

print("\nExact optimal selection:", best[2])
print("Optimal priority:", best[0])
print("Time used:", best[1])

assert best[2] == ["A", "B"]
assert best[0] == 153
assert best[1] == budget

print("\nPASS: real-candidate optimization validated.")


REAL-DATA OPTIMIZATION VALIDATION
---------------------------------------------
Selected: ['A', 'B'] | Priority: 153 | Time: 6
Selected: ['A', 'C'] | Priority: 125 | Time: 6
Selected: ['A'] | Priority: 100 | Time: 3
Selected: ['B', 'C'] | Priority: 78 | Time: 6
Selected: ['B'] | Priority: 53 | Time: 3
Selected: ['C'] | Priority: 25 | Time: 3
Selected: ['None'] | Priority: 0 | Time: 0

Exact optimal selection: ['A', 'B']
Optimal priority: 153
Time used: 6

PASS: real-candidate optimization validated.
